# Lab 7 — Full Engagement: The Autofill Decision
# المعمل 7 — دراسة حالة متكاملة: قرار الإكمال التلقائي

**Module 7 — Experimentation Case Study**
**الوحدة 7 — دراسة حالة تطبيقية في التجريب**

*SDA-DSC-213 · SDAIA Academy · Day 4, Hour 2 · 50 minutes · dress rehearsal for the capstone*

---

## The engagement

The Injaz Product & Policy Committee meets in ten days. Two decisions are on the agenda and
they arrive as one question:

> *"Should we launch AI-assisted form autofill nationally, and did last year's reminder pilot
> justify extending it?"*

Nobody labelled this "use DiD." One limb is a **future feature you can randomise**. The other
is a **past pilot you cannot**. Your value is choosing correctly, executing correctly, and
translating both into a recommendation a director can act on.

The engagement runs in a fixed order, and skipping steps is where credibility leaks:

```
1 FRAME  ->  2 DESIGN  ->  3 POWER  ->  [FREEZE]  ->  4 ANALYSE  ->  5 VALIDATE  ->  6 DECIDE
   \________ before you see any outcome ________/      \__ follow the frozen plan __/
```

| | |
|---|---|
| **Objective** | Sequence a full engagement, analyse the autofill A/B with CUPED and guardrails, reconcile it with the quasi-experimental evidence, and produce a decision memo |
| **Data** | `injaz_autofill_ab.csv` (70,000), `injaz_traffic_forecast.csv`, `injaz_regions_panel.csv`, `injaz_users.csv` |
| **Decision threshold** | **+1.0 pp** on completion — below that the build and inference cost is not recovered |
| **Graded skill** | **judgement**, not code |

### Bilingual key terms (first use)

| English | العربية |
|---|---|
| Frozen analysis plan | خطة تحليل مُجمّدة |
| Overall evaluation criterion (OEC) | معيار التقييم الشامل |
| Guardrail metric | مؤشر حماية |
| Practical significance | الدلالة العملية |
| Decision memo | مذكرة قرار |


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `causal_utils` importable from anywhere under course_assets/
ROOT = Path.cwd()
while not (ROOT / "causal_utils").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from causal_utils import use_sdaia_style
use_sdaia_style()

RANDOM_SEED = 213          # fixed for every lab: reproducibility is graded
rng = np.random.default_rng(RANDOM_SEED)
print("Environment ready ·", ROOT)

In [ ]:
# --- lab scaffolding: self-checks that never crash the notebook ----------
def check(name, fn):
    """Run a self-check. Prints PASS/FAIL/TODO instead of raising, so the
    notebook always runs top-to-bottom even with unfinished exercises."""
    try:
        fn()
    except AssertionError as e:
        print(f"[ TODO ]  {name}: {e}")
        return False
    except Exception as e:
        print(f"[ ERROR ] {name}: {type(e).__name__}: {e}")
        return False
    print(f"[  OK  ]  {name}")
    return True


def needs(*objs, msg="complete the TODO cell above first"):
    """Guard downstream cells so an unfinished TODO skips rather than crashes."""
    if any(o is None for o in objs):
        print(f"[ SKIPPED ] {msg}")
        return False
    return True

print("Scaffolding loaded: use check(...) to self-test and needs(...) to guard.")

## Step 0 — Method selection *(3 min)*

Before anything else: **which method, and why?** The selection follows the data situation,
never preference.

| Question | Data situation | Method |
|---|---|---|
| Can we randomise going forward? | yes → **A/B test** — always the first choice |
| Did a treatment turn on at a known time for some units? | yes → **difference-in-differences** |
| Is there an as-good-as-random source of variation? | yes → **instrumental variables** |
| Only cross-sectional data with measured confounders? | → **matching / weighting** + sensitivity |
| No measured way to close the backdoors? | → **declare it not identifiable** |

Run the drill below on six one-line Injaz questions before you touch the data.

In [ ]:
# TODO: assign the correct method to each question. Allowed values:
#   "ab_test", "did", "iv", "matching", "not_identifiable"
DRILL = {
    "Should we launch autofill nationally next quarter?":                       None,
    "Did last year's region-by-region reminder rollout raise completion?":       None,
    "Does filing online rather than in person raise completion?":                None,
    "Did the voluntary training programme help those who took it?":              None,
    "Do users who contact support complete more often, and should we push them?": None,
    "Does a redesigned upload step raise completion?":                            None,
}
ANSWER_KEY = ["ab_test", "did", "iv", "matching", "not_identifiable", "ab_test"]

def _c():
    got = list(DRILL.values())
    assert all(v is not None for v in got), "fill in all six"
    assert got == ANSWER_KEY, f"got {got}"
check("method-selection drill 6/6", _c)

## Step 1 — Frame and freeze the analysis plan *(8 min)*

Everything in this cell is committed **before any outcome is observed**. The freeze is the
whole point: metrics chosen after seeing data are not metrics, they are conclusions with a
p-value stapled on.

`freeze_plan()` timestamps and hashes the plan. The hash goes in the memo so a reader can
verify nothing moved.

In [ ]:
# TODO: complete the AnalysisPlan for the autofill experiment, then freeze it.
#   Required fields: experiment_name, decision, randomisation_unit, oec, guardrails
#   (at least 3), trigger_definition, unit_of_analysis, mde, practical_threshold.
#   The decision threshold for this feature is 1.0 pp = 0.010.
from causal_utils import AnalysisPlan, freeze_plan

plan = None      # <-- replace

def _c():
    assert plan is not None, "plan is still None"
    assert len(plan.guardrails) >= 3, "at least three guardrails are required"
    assert plan.practical_threshold == 0.010, "the decision threshold is 1.0 pp"
    assert plan.plan_hash is not None, "you must call freeze_plan(plan)"
check("analysis plan complete and frozen", _c)

## Step 2 — Power and sizing *(8 min)*

**Before** the data. Can the experiment we are about to run actually detect the effect we
care about? Three things bite teams here: *n* is per arm, not every eligible user triggers,
and a run length must be a whole number of weeks (KSA weekend traffic is a different
population).

**On the run-length floor.** `run_length` rounds up to whole weeks because weekday and
weekend users are different people: Friday and Saturday traffic is about 58% of a weekday's,
so a run that stops mid-week samples a biased slice. Here the arithmetic alone asks for
10 days, which rounds up to **14 days — two whole weeks**, the same floor Lab 3 imposed
explicitly. Read the plan below against that number, not against the raw arithmetic.

In [ ]:
# TODO: (a) baseline completion among triggered CONTROL users -> `baseline`
#       (b) required n per arm at the 1.0 pp MDE -> `n_required`
#       (c) run length from the traffic forecast -> `rl`
#       (d) the CUPED-adjusted alternative and its saving
from causal_utils import (sample_size_proportions, run_length, cuped_adjust,
                          power_for_n, mde_for_n)

ab = pd.read_csv(DATA / "injaz_autofill_ab.csv")
traffic = pd.read_csv(DATA / "injaz_traffic_forecast.csv")
triggered = ab[ab["triggered"] == 1].copy()

baseline = None
n_required = None
rl = None

def _c():
    assert baseline is not None and n_required is not None and rl is not None,         "fill in baseline / n_required / rl"
    assert 0.50 < baseline < 0.60, "baseline completion should be around 55%"
    assert 30_000 < n_required < 50_000, "n per arm at a 1.0 pp MDE should be ~39,000"
check("sizing computed", _c)

### Step 2b — CUPED saving, and the uncomfortable arithmetic

CUPED uses a **pre-treatment** covariate to strip variance out of the outcome. Variance falls
by ρ², so the effective sample rises — for free, and without bias, provided the covariate
cannot have been affected by treatment.

Then compare what you *needed* against what you *have*. This is the cell that determines the
verdict at the end of the lab, and most teams skip it.

In [ ]:
# TODO: (a) run cuped_adjust(triggered.completed, triggered.prior_completion_rate)
#       (b) compute the CUPED-adjusted required n and the % saving
#       (c) compare n_required against the ACTUAL per-arm sample
#       (d) compute the MDE this experiment can actually resolve: mde_for_n(n_actual, baseline)
cuped = None
n_actual = None
mde_actual = None

def _c():
    assert cuped is not None and n_actual is not None and mde_actual is not None,         "fill in cuped / n_actual / mde_actual"
    assert 0.12 < cuped.variance_reduction < 0.17, "CUPED should buy roughly 14%"
    assert mde_actual > plan.mde, ("this experiment cannot resolve its own threshold - "
                                   "that is the finding, not an error")
check("CUPED quantified; the experiment is under-run for its own threshold", _c)

## Step 3 — Experiment integrity: SRM and balance *(6 min)*

Before any effect estimate. An experiment that fails these is **invalid** and must not be
analysed — the correct response is to find the bug, not to adjust and continue.

In [ ]:
# TODO: (a) srm_check on the full assigned population and on the triggered subset
#       (b) balance_table on pre-treatment covariates within the triggered subset
#       Store the SRM results in `srm_all` / `srm_trig` and the table in `bal`.
from causal_utils import srm_check, balance_table

BALANCE_COVS = ["age", "device_age_years", "prior_sessions", "prior_completion_rate",
                "digital_literacy_score", "region", "device"]
srm_all = None
srm_trig = None
bal = None

def _c():
    assert srm_all is not None and srm_trig is not None and bal is not None,         "run all three checks"
    assert srm_all.passed and srm_trig.passed, "SRM failed - the experiment is invalid"
    assert bal["abs_smd"].max() < 0.10, "a covariate is imbalanced"
check("integrity: SRM passes and every |SMD| < 0.1", _c)

## Step 4 — Analyse: the OEC *(10 min)*

Now, and only now, look at outcomes — following the frozen plan, not improvising.

Three estimates to produce:

1. the **raw** difference on triggered users (the plan's OEC),
2. the **CUPED-adjusted** estimate with robust SEs — the plan's primary,
3. the **all-assigned** estimate, as a demonstration of the dilution trap.

In [ ]:
# TODO: (a) two_proportion_test on triggered users -> `eff_raw`
#       (b) attach cuped.y_adjusted to `triggered` as `completed_cuped`, then
#           regression_effect(...) with HC3 SEs -> `eff_cuped`
#       (c) two_proportion_test on ALL assigned users -> `eff_all`
from causal_utils import two_proportion_test, regression_effect

eff_raw = None
eff_cuped = None
eff_all = None

def _c():
    assert eff_raw is not None and eff_cuped is not None and eff_all is not None,         "produce all three estimates"
    assert eff_cuped.se < eff_raw.se, "CUPED should TIGHTEN the interval"
    assert eff_all.estimate < eff_raw.estimate, "the all-assigned estimate should be diluted"
check("OEC estimated three ways; CUPED tightens, all-assigned dilutes", _c)

## Step 5 — Guardrails, with the correct correction family *(6 min)*

Four guardrails were pre-registered. The family is four tests, so **Bonferroni** — you want to
be confident that *none* is broken (FWER), not that most are fine (FDR).

Direction matters as much as significance. A drop in `drop_off` is an *improvement*; a rise in
`support_contact` is a *harm*.

In [ ]:
# TODO: estimate the effect on each guardrail with regression_effect (HC3),
#       then apply correct_pvalues(..., method="bonferroni").
#       Store the estimates in `guardrail_effects` (list) and the table in `guardrails`.
from causal_utils import correct_pvalues

guardrail_effects = None
guardrails = None

def _c():
    assert guardrails is not None, "guardrails is still None"
    assert len(guardrails) == len(plan.guardrails), "one row per pre-registered guardrail"
    assert (guardrails["family_size"] == len(plan.guardrails)).all(), \
        "the correction family must be the pre-registered guardrail family"
check("guardrails checked with the pre-registered correction family", _c)

## Step 6 — Reconcile with the observational evidence *(8 min)*

The committee is deciding on a portfolio, not a feature. Bring in the two quasi-experimental
estimates from Labs 5a and 5b so the three sit on one axis with one threshold line.

**Triangulation, not cherry-picking.** Agreement across methods that fail in *different* ways
strengthens a claim. Disagreement is itself a finding — usually a violated assumption.

In [ ]:
# TODO: recompute the reminder DiD (Lab 5b) and the training ATT (Lab 5a) here,
#       so the memo is reproducible from ONE notebook. Store them in `did_reminders`
#       and `att_training`.
from causal_utils import did_twfe, match_nearest

did_reminders = None
att_training = None

def _c():
    assert did_reminders is not None and att_training is not None,         "recompute both prior estimates"
    assert 0.02 < did_reminders.att < 0.03, "reminder DiD should be about +2.5 pp"
    assert 0.05 < att_training.att < 0.07, "training ATT should be about +6.0 pp"
check("prior estimates reproduced inside this notebook", _c)

In [ ]:
# TODO: build the decision plot with decision_plot(estimates, threshold=plan.practical_threshold).
#       `estimates` is a list of objects with .label / .estimate (or .att) / .ci_low / .ci_high.
#       Include: the CUPED autofill estimate, the reminder DiD, the training ATT.
from causal_utils import decision_plot

print("[ TODO ] decision plot")

### Step 6b — Triangulation and the evidence ranking

Rank the three by **strength of evidence**, not by size of effect. The board will ask.

In [ ]:
RANKING = {
    "1_strongest": "TODO",
    "2_middle": "TODO",
    "3_weakest": "TODO",
    "why_ranking_not_effect_size": "TODO",
}
for k, v in RANKING.items():
    print(f"\n{k}: {v}")

## Step 7 — The verdict: statistical vs practical significance *(6 min)*

`practical_significance_verdict` maps a **confidence interval onto a decision**, not a p-value
onto a verdict. There are four cases and only one of them is *ship*.

| Interval | Verdict |
|---|---|
| entirely above the threshold | **SHIP** |
| entirely below zero | **STOP / ROLL BACK** |
| positive but entirely below the threshold | **HOLD** — real but too small |
| positive and **straddling** the threshold | **HOLD** — extend, or accept the risk |
| containing zero | **INCONCLUSIVE** |

In [ ]:
from causal_utils import practical_significance_verdict

extra_days = None


def explain_verdict():
    global extra_days
    print(practical_significance_verdict(eff_cuped, plan.practical_threshold,
                                         "Autofill completion (CUPED)"))
    print()
    print(practical_significance_verdict(eff_raw, plan.practical_threshold,
                                         "Autofill completion (raw, sanity check)"))

    print("\n" + "=" * 72)
    print("WHY THIS IS 'HOLD' AND NOT 'SHIP' - the reasoning, spelled out")
    print("=" * 72)
    print(f"""
  p-value        : {eff_cuped.p_value:.4f}   -> statistically significant. The effect
                   is real; it is not zero.
  effect         : {100*eff_cuped.estimate:+.2f} pp -> above the {100*plan.practical_threshold:.1f} pp threshold
                   AS A POINT ESTIMATE.
  interval       : [{100*eff_cuped.ci_low:+.2f}, {100*eff_cuped.ci_high:+.2f}] pp
                   -> and here is the problem. The lower bound, {100*eff_cuped.ci_low:+.2f} pp,
                   sits BELOW the {100*plan.practical_threshold:.1f} pp threshold.

  In plain words: the data establish that autofill helps. They do NOT establish
  that it helps ENOUGH. Effects from {100*eff_cuped.ci_low:+.2f} pp (well short of the bar) to
  {100*eff_cuped.ci_high:+.2f} pp (comfortably over it) are all consistent with what we observed.

  'p < 0.05, ship it' answers the question 'is there an effect?'. The committee
  asked 'is the effect big enough to justify the build?'. Those are different
  questions and this experiment answers only the first.

  And we KNEW this in Step 2, before any outcome was looked at: the run collected
  {n_actual:,} per arm against the {n_required:,} the design required, so the smallest
  resolvable effect was {100*mde_actual:.2f} pp against a {100*plan.practical_threshold:.1f} pp threshold. The
  interval had to straddle. This is an under-run experiment, not an ambiguous
  feature.

  RECOMMENDATION: HOLD and EXTEND. Keep the test running to the designed sample.
""")

    n_to_resolve = int(np.ceil(
        n_actual * (eff_cuped.se
                    / ((eff_cuped.estimate - plan.practical_threshold) / 1.96)) ** 2))
    extra_days = int(np.ceil((2 * (n_to_resolve - n_actual))
                             / (traffic["eligible_users"].mean()
                                * traffic["trigger_rate"].mean())))
    extra_days += (7 - extra_days % 7) % 7
    print(f"  HOW MUCH LONGER: to move the lower bound above {100*plan.practical_threshold:.1f} pp at the")
    print(f"  currently observed effect size, roughly {n_to_resolve:,} per arm is needed")
    print(f"  ({n_to_resolve/n_actual:.1f}x the current sample) - about {extra_days} more days")
    print("  of traffic, rounded to whole weeks. That is the number to put to the")
    print(f"  committee: not 'we need more data', but '{extra_days // 7} more weeks and you get")
    print("  a decision either way'.")

    def _c():
        assert eff_cuped.ci_low < plan.practical_threshold < eff_cuped.ci_high, \
            "the autofill CI should STRADDLE the 1.0 pp threshold"
        assert eff_cuped.p_value < 0.05, "the effect should be statistically significant"
    check("verdict is HOLD: significant, but the CI straddles the threshold", _c)


if needs(eff_cuped, eff_raw, plan, n_actual, mde_actual):
    explain_verdict()

## Step 8 — The decision memo *(8 min)*

**The memo is the deliverable, not the notebook.** Effect sizes and intervals in business
units, one recommendation per intervention, assumptions stated as risks, and an explicit
"what would change this."

No p-values reach this page. A director cannot act on 0.0001; they can act on "between +0.8
and +2.4 points, and we need at least +1.0."

In [ ]:
# TODO: render the memo with decision_memo(...). Required arguments:
#   title, decision, estimates (dict name -> object with .estimate/.ci_low/.ci_high),
#   threshold, recommendation, assumptions, risks, what_would_change_this, plan
from causal_utils import decision_memo

memo = None      # <-- replace

def _c():
    assert memo is not None, "memo is still None"
    assert "HOLD" in memo.upper(), "the recommendation for autofill must be HOLD"
    assert plan.plan_hash in memo, "the frozen-plan hash must appear in the memo"
    assert "p =" not in memo and "p-value" not in memo, "no p-values in a decision memo"
check("decision memo rendered with the frozen-plan hash and no p-values", _c)

if needs(memo):
    print(memo)

## Step 9 — Compare against the planted ground truth

In [ ]:
import json
GT = json.loads((DATA / "GROUND_TRUTH.json").read_text())


def ground_truth_comparison():
    ga, g5, gp = (GT["capstone_autofill"], GT["module5_user_cohort"],
                  GT["module5_regions_panel"])
    rows = [
        {"quantity": "Autofill lift (triggered, raw)", "recovered_pp": 100*eff_raw.estimate,
         "planted_pp": 100*ga["true_lift_on_exposed"]},
        {"quantity": "Autofill lift (triggered, CUPED)",
         "recovered_pp": 100*eff_cuped.estimate, "planted_pp": 100*ga["true_lift_on_exposed"]},
        {"quantity": "Autofill lift (all assigned - DILUTED)",
         "recovered_pp": 100*eff_all.estimate, "planted_pp": float("nan")},
        {"quantity": "Reminder ATT (TWFE)", "recovered_pp": 100*did_reminders.att,
         "planted_pp": 100*gp["true_ATT"]},
        {"quantity": "Training ATT (matching)", "recovered_pp": 100*att_training.att,
         "planted_pp": 100*g5["true_ATT_training"]},
        {"quantity": "CUPED variance reduction (%)",
         "recovered_pp": 100*cuped.variance_reduction,
         "planted_pp": GT["module3_pre_period"]["expected_cuped_variance_reduction_pct"]},
    ]
    out = pd.DataFrame(rows)
    out["gap"] = out["recovered_pp"] - out["planted_pp"]
    print(out.to_string(index=False, float_format=lambda v: f"{v:+.2f}"))

    print(f"""
THE PLANTED DESIGN OF THIS EXPERIMENT
--------------------------------------
The autofill effect was planted at {100*ga['true_lift_on_exposed']:.1f} pp against a
{ga['ship_threshold_pp']:.1f} pp decision threshold, and the sample was chosen so the
confidence interval would STRADDLE that threshold. The dataset note says it
plainly: "the effect is real and statistically significant, yet the data cannot
establish that it is large enough to justify the build."

That is not a flaw in the data. It is the most common real situation in
industrial experimentation and the one that separates a p-value reader from a
decision partner. The correct answer is HOLD / EXTEND.

  recovered CUPED effect : {100*eff_cuped.estimate:+.2f} pp
  interval               : [{100*eff_cuped.ci_low:+.2f}, {100*eff_cuped.ci_high:+.2f}] pp
  threshold              : {100*plan.practical_threshold:+.2f} pp   <- inside the interval
  reminder DiD           : {100*did_reminders.att:+.2f} pp (planted {100*gp['true_ATT']:+.1f} pp full
                           effect; TWFE averages in the two-month ramp-in, so it
                           correctly lands below the mature effect - see Lab 5b)""")

    def _c():
        assert abs(eff_raw.estimate - ga["true_lift_on_exposed"]) < 0.003, \
            "raw autofill lift should match the planted +1.4 pp"
        assert eff_cuped.ci_low < ga["ship_threshold_pp"]/100 < eff_cuped.ci_high, \
            "the CI must straddle the threshold - that is the designed lesson"
        assert abs(att_training.att - g5["true_ATT_training"]) < 0.01, "training ATT off"
    check("all three effects recovered; the autofill CI straddles the threshold", _c)
    return out


if needs(eff_raw, eff_cuped, eff_all, did_reminders, att_training, cuped):
    comparison = ground_truth_comparison()

## What you now own

The **engagement template**. This is what the capstone asks you to reproduce on your own,
and what SDA-DSC-311 (Decision Science) consumes as its input.

In [ ]:
def build_artefact():
    return {
        "notebook": "lab7 - full engagement: the autofill decision",
        "the_template": [
            "0. SELECT the method from the data situation, not from preference",
            "1. FRAME  - AnalysisPlan(...) + freeze_plan(...)  [before any outcome]",
            "2. POWER  - sample_size_proportions / run_length / cuped_adjust; and",
            "            compare what you NEEDED against what you HAVE (mde_for_n)",
            "3. VALIDATE INTEGRITY - srm_check on assigned AND on triggered; balance_table",
            "4. ANALYSE - the frozen OEC, CUPED-adjusted, robust SEs",
            "5. GUARDRAILS - the pre-registered family, the pre-registered correction",
            "6. RECONCILE - decision_plot + forest_plot against the threshold",
            "7. VERDICT - practical_significance_verdict: interval vs THRESHOLD",
            "8. MEMO   - decision_memo(...) with the plan hash, no p-values",
        ],
        "artefacts_for_the_capstone": [
            "frozen AnalysisPlan JSON + hash", "sizing table with the shortfall",
            "SRM + balance evidence", "guardrail table with the correction family",
            "decision plot vs threshold", "forest plot / triangulation",
            "one-page decision memo",
        ],
        "numbers_recovered": {
            "autofill_raw_pp": round(100*eff_raw.estimate, 2),
            "autofill_cuped_pp": round(100*eff_cuped.estimate, 2),
            "autofill_cuped_ci_pp": [round(100*eff_cuped.ci_low, 2),
                                     round(100*eff_cuped.ci_high, 2)],
            "autofill_all_assigned_pp": round(100*eff_all.estimate, 2),
            "threshold_pp": 100*plan.practical_threshold,
            "n_required_per_arm": n_required,
            "n_actual_per_arm": n_actual,
            "mde_resolvable_pp": round(100*mde_actual, 2),
            "cuped_variance_reduction": round(cuped.variance_reduction, 3),
            "reminders_did_pp": round(100*did_reminders.att, 2),
            "training_att_pp": round(100*att_training.att, 2),
            "verdict": "HOLD / EXTEND",
        },
        "rules_you_can_now_state": [
            "Steps 1-3 happen before you see outcomes. Step 4 follows the frozen plan.",
            "Compare the INTERVAL to the THRESHOLD, never the p-value to zero.",
            "'Not significant' is not 'no effect'; say what magnitudes remain compatible.",
            "Disagreement between methods is a finding, not an inconvenience.",
            "The memo is the deliverable. The notebook is the evidence behind it.",
            "A correct 'hold, keep running' is a better answer than a fast, wrong 'ship'.",
        ],
    }


if needs(eff_raw, eff_cuped, eff_all, did_reminders, att_training, cuped, mde_actual):
    ARTEFACT = build_artefact()
    for k, v in ARTEFACT.items():
        print(f"\n{k}:")
        print("  " + (("\n  ".join(str(x) for x in v)) if isinstance(v, list) else str(v)))

## Mini exercises

### 1. Debugging — the memo that says SHIP *(5 min)*

A colleague submits this recommendation:

> *"Autofill increases completion by 1.57 percentage points with p = 0.0001, comfortably
> significant at the 5% level. The point estimate exceeds our 1.0-point threshold. We
> recommend shipping to all users."*

Every number in it is correct. The recommendation is wrong. **Rewrite it.**

In [ ]:
# TODO: write the corrected recommendation in REWRITE (3-5 sentences).
#       It must: lead with effect + interval in business units, compare the LOWER
#       BOUND to the threshold, give a clear verdict, and say what would change it.
REWRITE = """TODO"""
print(REWRITE)

# TODO: also name the specific error in one line.
ERROR_NAME = "TODO"
print("\nError:", ERROR_NAME)

### 2. Case study — "The Board Wants One Number" *(6 min)*

The steering board asks for **one figure** for the impact of last year's combined digital-
experience investments, to justify next year's budget.

Collapsing three estimands on three populations into one effect size would be misleading. But
"we can't" is not an acceptable answer to a board. Find the legitimate single figure.

In [ ]:
# TODO: compute TOTAL ATTRIBUTABLE COMPLETIONS with a range - a legitimate single
#       figure. For each intervention: effect (and its CI) x the affected population.
#       Use the population sizes below. Store the result in `attributable` (a DataFrame)
#       and write BOARD_ANSWER.
POPULATIONS = {
    "Autofill (if launched, triggered users/yr)": 2_400_000,
    "Reminders (activated regions, users/yr)": 5_100_000,
    "Training (enrollees/yr)": 180_000,
}
attributable = None
BOARD_ANSWER = "TODO"

if needs(attributable):
    print(attributable.to_string(index=False))
    print(BOARD_ANSWER)

### 3. Discussion *(4 min)*

**(a)** When is a correct "hold and keep running" more valuable than a fast "ship"?

**(b)** How do you present a non-identifiable result so leadership still trusts the analysis?

In [ ]:
DISCUSSION = {"a_hold_vs_ship": "TODO", "b_non_identifiable": "TODO"}
for k, v in DISCUSSION.items():
    print(f"\n{k}:\n  {v}")

---

### Quiz recall (Module 7)

1. **First question when choosing a method?** → can we randomise going forward?
2. **Which steps must precede seeing outcomes?** → framing, design, powering, and freezing
   the plan.
3. **How should an effect be reported to leadership?** → effect size with a confidence
   interval in business units, plus a practical-significance verdict.
4. **What when experimental and observational estimates disagree?** → reconcile via the likely
   violated assumption; report both, do not cherry-pick.
5. **Is "we cannot conclude" ever a correct deliverable?** → yes, when justified, with what
   was ruled out and the concrete next step.

### Next

**The capstone.** Open `capstone_starter.ipynb`. It is this engagement, run by you, on the
full brief — plus one extension of your choice. Reuse everything in `ARTEFACT` above; that is
what it is for.